# Make notebook experiments reproducible and reviewable

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Record a minimal environment fingerprint, isolate random state, hash a synthetic input, and test a bootstrap confidence interval with reproducible resampling.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses Python standard library, NumPy and pandas. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Capture a safe environment fingerprint

Record versions that explain numerical behavior without dumping environment variables, home directories, package indexes or secrets. The collection's dependency lock captures the full installation; this small table is a human-readable runtime check.

In [ ]:
import json
import sys
from hashlib import sha256
from importlib.metadata import version

import numpy as np
import pandas as pd

fingerprint = {
    "python": ".".join(map(str, sys.version_info[:3])),
    "numpy": version("numpy"),
    "pandas": version("pandas"),
}
pd.DataFrame([fingerprint])

## 2. Canonicalize the experiment inputs

A fixed seed alone is insufficient if the data or algorithm changes. Serialize the full authored fixture deterministically, then retain a SHA-256 digest next to the configuration. These are invented latency measurements in milliseconds.

In [ ]:
configuration = {"seed": 17, "resamples": 5000, "confidence": 0.95}
measurements = [11.0, 12.0, 12.5, 13.0, 13.5, 14.0, 15.0, 15.5]
input_bytes = json.dumps(
    {"configuration": configuration, "measurements": measurements},
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
).encode()
input_digest = sha256(input_bytes).hexdigest()
assert len(input_digest) == 64
print({"input_sha256": input_digest, "configuration": configuration})

## 3. Estimate a mean with a percentile bootstrap interval

Each resample draws the same number of observations with replacement. The 2.5th and 97.5th percentiles form a 95% percentile interval for this example. The method assumes the observations are representative and independent; a deterministic result does not establish those assumptions.
Normalize observations by their largest absolute value before averaging or interpolating quantiles, then restore the original units. This avoids overflow when individually finite observations have a sum outside float64 range. It does not make float64 exact arithmetic; values much smaller than the scale may lose relative precision.


In [ ]:
def bootstrap_mean(values, seed, resamples=5000, confidence=0.95):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < 2 or not np.isfinite(x).all():
        raise ValueError("Need at least two finite one-dimensional observations")
    if type(resamples) is not int or resamples < 100 or not 0 < confidence < 1:
        raise ValueError("Invalid bootstrap configuration")
    # Compute in bounded, dimensionless units before restoring the original scale.
    scale = float(np.max(np.abs(x)))
    scaled = x / scale if scale else x.copy()
    # A local generator avoids dependence on notebook cell execution history.
    generator = np.random.Generator(np.random.PCG64(seed))
    means = np.empty(resamples)
    # One resample at a time keeps temporary allocation proportional to input length.
    for index in range(resamples):
        means[index] = generator.choice(scaled, size=len(x), replace=True).mean()
    tail = (1 - confidence) / 2
    low, high = np.quantile(means, [tail, 1 - tail])
    result = {
        "mean": float(scaled.mean() * scale),
        "low": float(low * scale),
        "high": float(high * scale),
    }
    if not np.isfinite(list(result.values())).all():
        raise ValueError("Bootstrap result exceeded the supported floating-point range")
    return result


result = bootstrap_mean(measurements, **configuration)
assert result["low"] <= result["mean"] <= result["high"]
pd.DataFrame([result])

## 4. Prove replay and failure invariants

Exact replay is checked within the same tested environment. Cross-version numerical results may differ, which is why the lock and version fingerprint matter. Constant input should produce a zero-width interval.

In [ ]:
assert bootstrap_mean(measurements, **configuration) == result
constant = bootstrap_mean([7, 7, 7], **configuration)
assert constant == {"mean": 7.0, "low": 7.0, "high": 7.0}
# Finite inputs can overflow a naive sum; check both extreme scales and signs.
for value in (1e308, np.finfo(float).max, np.nextafter(0.0, 1.0)):
    extreme_constant = bootstrap_mean([value, value], seed=17, resamples=100)
    assert extreme_constant == {"mean": value, "low": value, "high": value}
extreme_signed = bootstrap_mean([-1e308, 1e308], seed=17, resamples=100)
assert extreme_signed == {"mean": 0.0, "low": -1e308, "high": 1e308}
assert bootstrap_mean([0.0, 0.0], seed=17, resamples=100) == {"mean": 0.0, "low": 0.0, "high": 0.0}
assert result["mean"] == 13.3125
for values in [[], [1], [1, np.nan], [[1, 2], [3, 4]]]:
    try:
        bootstrap_mean(values, **configuration)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid measurements accepted")
print("Replay, extreme-scale arithmetic, constant-input and validation checks passed.")

## 5. Assemble a portable experiment record

Save this structure alongside a real experiment only after reviewing it for sensitive inputs. Here it remains in memory. A dependency audit must be rerun near publication; a clean audit on one date is not a permanent guarantee.

In [ ]:
experiment = {
    "environment": fingerprint,
    "input_sha256": input_digest,
    "configuration": configuration,
    "result": result,
    "data_provenance": "Newly authored synthetic latency fixture",
}
encoded = json.dumps(experiment, sort_keys=True, allow_nan=False)
assert json.loads(encoded) == experiment
experiment

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

Eight invented observations cannot support a real performance claim. For correlated measurements use an appropriate resampling design; for small or skewed samples evaluate interval coverage rather than assuming the percentile bootstrap is accurate. Seeds, lockfiles and hashes improve repeatability but do not prove correctness or eliminate dependency risk.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [NumPy random generator and compatibility policy](https://numpy.org/doc/stable/reference/random/compatibility.html)
- [NumPy quantile](https://numpy.org/doc/stable/reference/generated/numpy.quantile.html)
- [Python package metadata](https://docs.python.org/3/library/importlib.metadata.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.